# 1.初期工作准备

In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate
)
from pyexpat import features

from notebook.基础.model_practice import response

In [5]:
# 配置环境变量
load_dotenv()

True

In [8]:
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
# 验证环境变量
if not api_key or not base_url:
    raise ValueError(
        "\n请先在 .env 文件中设置有效的 DASHSCOPE_API_KEY\n"
        "访问 https://console.groq.com/keys 获取免费密钥"
    )
# 初始化模型
model = init_chat_model(
    model="qwen-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

# 2.prompt_template和chat_prompt_template教学

In [21]:
def example_prompt_template_basics():
    print("\n" + "="*70)
    print("PromptTemplate 基础用法")
    print("="*70)

    print("方式一：form_template（推荐）")
    template1 = PromptTemplate.from_template(
        "将以下文本翻译成{language}：\n{text}"
    )
    prompt1 = template1.format(language="中文", text="I love programming.")
    print(f"生成的提示词：\n{prompt1}\n")
    response1 = model.invoke(prompt1)
    print(f"模型输出：\n{response1.content}\n")

    print("方式二：显示指定变量")
    template2 = PromptTemplate(
        input_variables=["product", "feature"],
        template="为{product}产品写一句广告词，特点是{feature}"
    )
    prompt2 = template2.format(product="手机", feature="三屏幕可折叠")
    print(f"生成的提示词：\n{prompt2}\n")
    response2 = model.invoke(prompt2)
    print(f"模型输出：\n{response2.content}\n")

    print("方式三：使用 invoke（更方便）")
    template3 = PromptTemplate.from_template(
        "写一首关于{theme}的{style}风格的诗，不超过4行。"
    )
    prompt_value = template3.invoke(
        {
            "theme": "旅行",
            "style": " Classic"
        }
    )
    print(f"生成的提示词：\n{prompt_value.text}\n")
    response3 = model.invoke(prompt_value.text)
    print(f"模型输出：\n{response3.content}\n")

In [11]:
def example_chat_prompt_template():
    print("\n" + "="*70)
    print("ChatPromptTemplate 基础用法")
    print("="*70)

    print("方式一：元组格式（推荐）")
    chat_template = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}：擅长{expertise}。"),
        ("user", "请帮我{task}")
    ])
    print(f"模型变量: {chat_template.input_variables}")
    messages = chat_template.format_messages(
        role="Python 导师",
        expertise="用简单的方式解释复杂概念",
        task="解释什么是列表推导式"
    )
    print(f"生成的消息：")
    for message in messages:
        print(f"{message.type}: {message.content}")
    response = model.invoke(messages)
    print(f"模型输出：\n{response.content[:150]}...\n")

    print("方式二：字符串简写")
    simple_template = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}：擅长{expertise}。"),
        ("user", "请帮我{task}")
    ])
    messages = simple_template.format_messages(
        role="Python 导师",
        expertise="用简单的方式解释复杂概念",
        task="解释什么是列表推导式"
    )
    response = model.invoke(messages)
    print(f"模型输出：\n{response.content[:100]}...\n")

# 3.多轮对话模板

In [26]:
def example_conversation_template():
    print("\n" + "="*70)
    print(" ConversationTemplate 示例")
    print("="*70)

    chat_template = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}：{instruction}。"),
        ("user", "请回答问题：{question1}"),
        ("assistant", "{answer1}"),
        ("user", "请回答问题：{question2}"),
    ])

    print("模板结构：")
    print("  1. System: 设定角色和指令")
    print("  2. User: 第一个问题")
    print("  3. Assistant: 第一个回答")
    print("  4. User: 第二个问题（基于上下文）\n")

    messages = chat_template.format_messages(
        role="Python 导师",
        instruction="用简单方式解释复杂概念",
        question1="请解释什么是列表推导式",
        answer1="列表推导式是一种简洁的方式，用于创建列表。",
        question2="请解释什么是元组"
    )
    print(f"生成的完整对话：")
    for i, msg in enumerate(messages, 1):
        content_preview = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
        print(f"{i}. {msg.type}: {content_preview}")

    response = model.invoke(messages)
    print(f"模型输出：\n{response.content}\n")

# 4.部分变量 预填充某些变量

In [31]:
def example_partial_variables():
    print("\n" + "="*70)
    print(" 部分变量 预填充某些变量")
    print("="*70)

    original_template = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}，你的目标用户是{audience}。"),
        ("user", "请{task}")
    ])
    print(f"原始模板变量：{original_template.input_variables}")
    partial_template = original_template.partial(
        role="Python 导师",
        audience="程序员"
    )
    print(f"部分变量填充后变量：{partial_template.input_variables}")
    messages1 = partial_template.format_messages(
        task="解释什么是列表推导式"
    )
    response1 = model.invoke(messages1)
    print(f"文章 1：{response1.content[:150]}...\n")

    # 复用模板，不同的 task
    messages2 = partial_template.format_messages(
        task="写一篇关于异步编程的文章开头"
    )

    response2 = model.invoke(messages2)
    print(f"文章 2：{response2.content[:150]}...\n")


# 5.模板+模型链式调用

In [35]:
def example_lcel_chains():
    print("\n" + "="*70)
    print(" 模板+模型链式调用")
    print("="*70)

    template = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}"),
        ("user", "{input}")
    ])

    # 使用 | 运算符创建链
    chain = template | model

    print("链的组成：")
    print("模板 | 模型")
    print("(Template) | (LLM)\n")

    response = chain.invoke({
        "role": "Python 导师",
        "input": "请解释什么是列表推导式"
    })
    print(f"模型输出：\n{response.content}\n")


In [36]:
def main():
    """运行所有示例"""
    print("\n" + "="*70)
    print(" LangChain 1.0 基础教程 - 提示词模板")
    print("="*70)

    try:
        # example_prompt_template_basics()
        # input("\n按 Enter 继续...")
        #
        # example_chat_prompt_template()
        # input("\n按 Enter 继续...")

        # example_conversation_template()
        # input("\n按 Enter 继续...")

        # example_partial_variables()
        # input("\n按 Enter 继续...")

        example_lcel_chains()

        print("\n" + "="*70)
        print(" 所有示例运行完成！")
        print("="*70)

    except KeyboardInterrupt:
        print("\n\n程序被用户中断")
    except Exception as e:
        print(f"\n运行出错：{e}")
        import traceback
        traceback.print_exc()

In [37]:
if __name__ == "__main__":
    main()


 LangChain 1.0 基础教程 - 提示词模板

 模板+模型链式调用
链的组成：
模板 | 模型
(Template) | (LLM)

模型输出：
列表推导式（List Comprehension）是 Python 中一种非常简洁且强大的创建列表的方式。它允许你用一行代码来生成一个列表，而不需要使用多行的 for 循环和条件语句。列表推导式的语法非常直观，通常也比传统的循环方式更易于阅读和理解。

### 基本语法
列表推导式的基本语法如下：

```python
[expression for item in iterable if condition]
```

- `expression`：对每个元素进行操作的表达式。
- `item`：迭代变量，表示从 `iterable` 中取出的每一个元素。
- `iterable`：一个可以迭代的对象，如列表、元组、字符串等。
- `if condition`：可选的条件表达式，用于过滤元素。

### 示例

1. **基本用法**：
   生成一个包含 0 到 9 的平方数的列表。

   ```python
   squares = [x**2 for x in range(10)]
   print(squares)  # 输出: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
   ```

2. **带条件的列表推导式**：
   生成一个包含 0 到 9 中偶数的平方数的列表。

   ```python
   even_squares = [x**2 for x in range(10) if x % 2 == 0]
   print(even_squares)  # 输出: [0, 4, 16, 36, 64]
   ```

3. **嵌套的列表推导式**：
   生成一个包含所有可能的 (x, y) 对的列表，其中 x 和 y 都在 0 到 2 之间。

   ```python
   pairs = [(x, y) for x in range(3) for y in range(3)]
   print(pairs)
   # 输出: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2,